# Using NCI Imaging Data Commons with MONAI

Copyright 2026 Imaging Data Commons

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

---

This tutorial demonstrates how to use imaging data from the [NCI Imaging Data Commons (IDC)](https://portal.imaging.datacommons.cancer.gov/) with MONAI for medical imaging AI tasks.

**IDC** is a cloud-based repository of publicly available cancer imaging data with:
- 50+ TB of radiology and pathology images
- No authentication required for access
- AI-generated and expert annotations
- All data in DICOM format

## Setup environment

In [ ]:
!pip install -q monai idc-index itk nibabel

# Restart runtime after installing ITK (required for ITK to load properly)
import sys
if "google.colab" in sys.modules:
    try:
        import itk
    except ImportError:
        print("Restarting runtime to load ITK...")
        import os
        os.kill(os.getpid(), 9)

## Setup imports

In [ ]:
import os
import tempfile

import torch
import numpy as np
import matplotlib.pyplot as plt
from idc_index import IDCClient

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
)
from monai.data import Dataset, DataLoader
from monai.data.image_reader import ITKReader
import monai

monai.config.print_config()

## 1. Query IDC Data

Use `idc-index` to query IDC's metadata index using SQL.

In [ ]:
# Initialize IDC client
client = IDCClient()

# Check IDC version and data scale
print(f"IDC version: {client.get_idc_version()}")

stats = client.sql_query("""
    SELECT COUNT(DISTINCT collection_id) as collections,
           COUNT(DISTINCT PatientID) as patients
    FROM index
""")
print(f"Collections: {stats.iloc[0]['collections']}, Patients: {stats.iloc[0]['patients']}")

In [ ]:
# Find lung CT collections by joining collections_index with index
client.fetch_index("collections_index")

# Join to get modality information (not available in collections_index directly)
lung_collections = client.sql_query("""
    SELECT c.collection_id, c.Subjects, c.CancerTypes,
           COUNT(DISTINCT CASE WHEN i.Modality = 'CT' THEN i.SeriesInstanceUID END) as ct_series
    FROM collections_index c
    JOIN index i ON c.collection_id = i.collection_id
    WHERE c.CancerTypes LIKE '%Lung%'
    GROUP BY c.collection_id, c.Subjects, c.CancerTypes
    HAVING ct_series > 0
    ORDER BY c.Subjects DESC
    LIMIT 5
""")
print("Lung CT collections:")
print(lung_collections.to_string(index=False))

In [ ]:
# Query specific CT series (using small rider_pilot collection for demo)
series_df = client.sql_query("""
    SELECT SeriesInstanceUID, PatientID, Modality,
           ROUND(series_size_MB, 2) as size_mb
    FROM index
    WHERE collection_id = 'rider_pilot' AND Modality = 'CT'
    LIMIT 3
""")
print(f"Found {len(series_df)} CT series")

## 2. Download DICOM Data

In [ ]:
# Download to temporary directory
data_dir = tempfile.mkdtemp(prefix="idc_monai_")
series_uids = list(series_df['SeriesInstanceUID'])

print(f"Downloading {len(series_uids)} series...")
client.download_from_selection(
    seriesInstanceUID=series_uids,
    downloadDir=data_dir,
    dirTemplate="%SeriesInstanceUID"
)
print("Done!")

## 3. Load with MONAI Transforms

MONAI's `LoadImaged` with `ITKReader` directly loads DICOM series from directories.

In [ ]:
# Define transforms for CT preprocessing
# Use ITKReader explicitly to load DICOM series from directories
transforms = Compose([
    LoadImaged(keys=["image"], reader=ITKReader()),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=(1.5, 1.5, 2.0)),
    ScaleIntensityRanged(keys=["image"], a_min=-175, a_max=250,
                         b_min=0.0, b_max=1.0, clip=True),
])

# Create dataset
data_dicts = [{"image": os.path.join(data_dir, uid)} for uid in series_uids]
dataset = Dataset(data=data_dicts, transform=transforms)

# Load sample
sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Value range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")

## 4. Visualize

In [ ]:
image = sample['image'][0]
z = image.shape[2] // 2

plt.figure(figsize=(6, 6))
plt.imshow(image[:, :, z].T, cmap='gray', origin='lower')
plt.title(f'CT from IDC (slice {z})')
plt.axis('off')
plt.show()

## 5. Find Paired Segmentations

IDC contains DICOM Segmentation objects. Use `seg_index` to find paired data.

In [ ]:
# Fetch seg_index
client.fetch_index("seg_index")

# Find CT with TotalSegmentator segmentations
paired = client.sql_query("""
    SELECT src.SeriesInstanceUID as image_uid,
           seg.SeriesInstanceUID as seg_uid,
           src.collection_id, seg.total_segments
    FROM seg_index seg
    JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
    WHERE src.Modality = 'CT'
      AND seg.AlgorithmName LIKE '%TotalSegmentator%'
    LIMIT 3
""")
print("CT with TotalSegmentator (104 structures):")
print(paired.to_string(index=False))

## 6. Check Licenses

In [ ]:
# Always check licenses before use
uid_list = ", ".join(f"'{uid}'" for uid in series_uids)
licenses = client.sql_query(f"""
    SELECT license_short_name, COUNT(*) as count
    FROM index WHERE SeriesInstanceUID IN ({uid_list})
    GROUP BY license_short_name
""")
print("Licenses:")
print(licenses.to_string(index=False))

## 7. Cleanup

In [ ]:
# import shutil; shutil.rmtree(data_dir)  # Uncomment to delete
print(f"Data at: {data_dir}")

## Summary

This tutorial demonstrated:
1. Querying IDC with `idc-index` SQL interface
2. Downloading DICOM data (no authentication needed)
3. Loading directly into MONAI with `LoadImaged`
4. Finding paired segmentations via `seg_index`

**Resources:**
- [IDC Portal](https://portal.imaging.datacommons.cancer.gov/)
- [IDC Documentation](https://learn.canceridc.dev/)
- [idc-index](https://github.com/ImagingDataCommons/idc-index)
- [IDC-MONAI toolkit](https://github.com/ImagingDataCommons/idc-monai)